# 03_graph_memory — what a built graph costs

A built graph used to spend most of its memory on 64 indexes and 21 primary keys, though no diagram query reads through an index. The graph now keeps neither, and a duplicate node id is refused by name.


Use the repository environment for this branch. Run these cells in order in a fresh Python kernel.


In [0]:
from __future__ import annotations

from types import SimpleNamespace
from typing import Any

from fastapi.testclient import TestClient

from maxitor.front.api.app import create_app
from maxitor.front.api.graph_store import GraphEntry, GraphStore
from maxitor.front.model.diagrams.resources.duckdb_graph_resource import DuckDBGraphResource

SOURCE = "https://duplicate.test/graph-json"
EMPTY_SIDEBAR = SimpleNamespace(level1_nodes=[], level2_diagrams=[], level2_nodes=[], level3_diagrams=[])
MIB = 1024 * 1024


def domain(node_id: str) -> dict[str, Any]:
    return {"id": node_id, "label": node_id, "type": "Domain", "properties": {"name": node_id, "description": ""}}


def lifecycle(node_id: str) -> dict[str, Any]:
    return {"id": node_id, "label": node_id, "type": "Lifecycle", "properties": {"field_name": "status"}}


def state(node_id: str, state_key: str) -> dict[str, Any]:
    return {"id": node_id, "label": state_key, "type": "StateInitial", "properties": {"lifecycle_class_id": "x.Status", "state_key": state_key}}


def payload(*nodes: dict[str, Any], edges: tuple[dict[str, Any], ...] = ()) -> dict[str, Any]:
    return {"schema_version": "1.0", "nodes": list(nodes), "edges": list(edges)}


def tiny() -> dict[str, Any]:
    lifecycle_id = "e.x.Entity:lifecycle:status"
    return payload(
        domain("d.default.Dom"),
        lifecycle(lifecycle_id),
        state(f"{lifecycle_id}:draft", "draft"),
        edges=(
            {"source_id": lifecycle_id, "target_id": f"{lifecycle_id}:draft", "type": "lifecycle_contains_state", "relationship": "COMPOSITION", "is_dag": False, "properties": {"state_key": "draft"}},
        ),
    )


## 1. What a built graph costs

The categories come from DuckDB's own `duckdb_memory()`. The catalogue shows no explicit index and no primary key.


In [0]:
graph = DuckDBGraphResource.build_from_json(tiny())
rows = graph.execute_fetch_dicts("SELECT tag, memory_usage_bytes FROM duckdb_memory() WHERE memory_usage_bytes > 0")
total = sum(int(row["memory_usage_bytes"]) for row in rows) / MIB
for row in sorted(rows, key=lambda item: str(item["tag"])):
    print(f"{row['tag']:<20} {int(row['memory_usage_bytes']) / MIB:.2f} MiB")
print(f"{'total':<20} {total:.2f} MiB")
indexes = graph.execute_fetch_dicts("SELECT index_name FROM duckdb_indexes()")
keys = graph.execute_fetch_dicts("SELECT constraint_name FROM duckdb_constraints() WHERE constraint_type = 'PRIMARY KEY'")
print(f"indexes: {len(indexes)}, primary keys: {len(keys)}")


IN_MEMORY_TABLE      0.57 MiB
total                0.57 MiB
indexes: 0, primary keys: 0


## 2. A duplicate node is refused by name

One Domain id listed twice is refused after loading, with the id and the table in the message.


In [0]:
duplicate = payload(domain("d.dup.Dom"), domain("d.dup.Dom"))
try:
    DuckDBGraphResource.build_from_json(duplicate)
except ValueError as error:
    print(f"ValueError: {error}")


ValueError: Graph has a duplicate node id "d.dup.Dom" in table "domain".


## 3. The same duplicate through Maxitor

The store wraps the build error, the middleware answers 422 with the reason, and nothing is kept for that source.


In [0]:
async def builder(source: str) -> GraphEntry:
    return GraphEntry(source=source, graph=DuckDBGraphResource.build_from_json(duplicate), sidebar=EMPTY_SIDEBAR, last_read=0.0)

app = create_app()
store = GraphStore(builder)
app.state.graph_store = store
client = TestClient(app, raise_server_exceptions=False)
response = client.get("/api/v1/list-domains", params={"source": SOURCE})
print(f"GET /api/v1/list-domains -> {response.status_code} {response.json()}")
print(f"stored for that source: {store.held} graphs")


GET /api/v1/list-domains -> 422 {'detail': 'Graph has a duplicate node id "d.dup.Dom" in table "domain".'}
stored for that source: 0 graphs


## 4. The same id in two tables, and an empty graph

The check keeps the primary key's old scope: a repeat across different node tables is accepted. An empty graph builds and answers with empty views.


In [0]:
shared = DuckDBGraphResource.build_from_json(payload(domain("shared.id"), lifecycle("shared.id")))
print(f"domain + lifecycle -> {len(shared.execute_fetch_dicts('SELECT id FROM nodes'))} nodes")
empty = DuckDBGraphResource.build_from_json(payload())
print(f"empty graph -> {len(empty.execute_fetch_dicts('SELECT id FROM nodes'))} nodes")


domain + lifecycle -> 2 nodes
empty graph -> 0 nodes
